# DB생성

기존 CSV 4개를 읽어서 SQLite DB 생성.

테이블 구조 유지

preprocessing_code/data/Emergency_shelter.db 생성


In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import (
    Column,
    DateTime,
    Float,
    ForeignKey,
    Integer,
    MetaData,
    String,
    Table,
    Text,
    UniqueConstraint,
    create_engine,
    insert,
)


def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "preprocessing_data" / "preprocessing").exists():
            return path
    raise FileNotFoundError("Could not find project root")


def clean_text(value):
    if pd.isna(value):
        return None
    value = " ".join(str(value).split())
    return value or None


def clean_frame(frame):
    frame = frame.copy()
    for column in frame.select_dtypes(include="object").columns:
        frame[column] = frame[column].map(clean_text)
    return frame


def shorten_sigungu(name):
    if name is None:
        return None

    for suffix in ["특별자치시", "특별자치도", "광역시", "특별시", "시", "군", "구"]:
        if name.endswith(suffix):
            return name[: -len(suffix)]

    return name


def first_notnull(series):
    for value in series:
        if pd.notna(value):
            return value
    return None


def merge_shelter_types(series):
    types = []
    for value in series:
        value = clean_text(value)
        if value is None:
            continue
        for item in value.split(","):
            item = clean_text(item)
            if item and item not in types:
                types.append(item)
    return ",".join(types) or None


def to_records(frame):
    return frame.astype(object).where(pd.notna(frame), None).to_dict(orient="records")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "preprocessing_data" / "preprocessing"
DB_PATH = PROJECT_ROOT / "preprocessing_code" / "data" / "Emergency_shelter.db"

danger = clean_frame(pd.read_csv(DATA_DIR / "danger_clean.csv"))
shelters_source = clean_frame(pd.read_csv(DATA_DIR / "final_shelter_dataset.csv"))
earthquake_source = clean_frame(pd.read_csv(DATA_DIR / "earthquake_shelter_clean_2.csv"))
tsunami_source = clean_frame(pd.read_csv(DATA_DIR / "tsunami_shelter_clean_2.csv"))

danger["발표시간"] = pd.to_datetime(danger["발표시간"], errors="coerce")

for frame in [shelters_source, earthquake_source, tsunami_source]:
    frame["위도"] = pd.to_numeric(frame["위도"], errors="coerce")
    frame["경도"] = pd.to_numeric(frame["경도"], errors="coerce")
    frame["수용인원"] = pd.to_numeric(frame["수용인원"], errors="coerce").round().astype("Int64")


In [2]:
shelter_regions = pd.concat(
    [
        shelters_source[["시도", "시군구"]],
        earthquake_source[["시도", "시군구"]],
        tsunami_source[["시도", "시군구"]],
    ],
    ignore_index=True,
).dropna(subset=["시도", "시군구"]).drop_duplicates()

region_name_map = {}
for sido, sigungu in shelter_regions.itertuples(index=False, name=None):
    region_name_map[(sido, shorten_sigungu(sigungu))] = sigungu

region_override_map = {
    ("경북", "군위"): ("대구", "군위군"),
}

mapped_regions = []
for sido, sigungu in danger[["지역", "시군구"]].itertuples(index=False, name=None):
    if (sido, sigungu) in region_override_map:
        mapped_regions.append(region_override_map[(sido, sigungu)])
        continue

    if sigungu is None or sido == sigungu:
        mapped_regions.append((sido, None))
        continue

    short_name = shorten_sigungu(sigungu)
    mapped_sigungu = region_name_map.get((sido, short_name), sigungu)
    mapped_regions.append((sido, mapped_sigungu))

danger[["매핑시도", "매핑시군구"]] = pd.DataFrame(
    mapped_regions,
    columns=["매핑시도", "매핑시군구"],
)

danger_regions = danger[["매핑시도", "매핑시군구"]].rename(
    columns={"매핑시도": "시도", "매핑시군구": "시군구"}
)

regions = pd.concat([shelter_regions, danger_regions], ignore_index=True)
regions["시군구"] = regions["시군구"].where(regions["시군구"].notna(), None)
regions = regions.drop_duplicates().sort_values(["시도", "시군구"], na_position="first").reset_index(drop=True)
regions.insert(0, "지역_id", range(1, len(regions) + 1))

region_id_map = {
    (row.시도, row.시군구): row.지역_id
    for row in regions.itertuples(index=False)
}

disaster_types = danger[["재난종류"]].drop_duplicates().sort_values("재난종류").reset_index(drop=True)
disaster_types = disaster_types.rename(columns={"재난종류": "재난이름"})
disaster_types.insert(0, "재난유형_id", range(1, len(disaster_types) + 1))

disaster_type_id_map = dict(zip(disaster_types["재난이름"], disaster_types["재난유형_id"]))

shelters = shelters_source.copy()
shelters["지역_id"] = [
    region_id_map[(sido, sigungu)]
    for sido, sigungu in shelters[["시도", "시군구"]].itertuples(index=False, name=None)
]
shelters = shelters.rename(columns={"지역": "지역설명"})
shelters = (
    shelters.groupby(["대피소명", "주소"], as_index=False)
    .agg(
        지역_id=("지역_id", "first"),
        대피소유형=("대피소유형", merge_shelter_types),
        위도=("위도", "first"),
        경도=("경도", "first"),
        지역설명=("지역설명", first_notnull),
        수용인원=("수용인원", "max"),
    )
)
shelters.insert(0, "대피소_id", range(1, len(shelters) + 1))

shelter_id_map = {
    (row.대피소명, row.주소): row.대피소_id
    for row in shelters.itertuples(index=False)
}

danger_alerts = danger.copy()
danger_alerts["지역_id"] = [
    region_id_map[(sido, sigungu)]
    for sido, sigungu in danger_alerts[["매핑시도", "매핑시군구"]].itertuples(index=False, name=None)
]
danger_alerts["재난유형_id"] = danger_alerts["재난종류"].map(disaster_type_id_map)
danger_alerts = danger_alerts[["발표시간", "지역_id", "재난유형_id", "특보등급", "해당지역"]].copy()
danger_alerts.insert(0, "위험정보_id", range(1, len(danger_alerts) + 1))

earthquake_links = earthquake_source[["대피소명", "주소", "수용인원"]].copy()
earthquake_links["대피소_id"] = [
    shelter_id_map[(name, address)]
    for name, address in earthquake_links[["대피소명", "주소"]].itertuples(index=False, name=None)
]
earthquake_links = earthquake_links[["대피소_id", "수용인원"]].drop_duplicates().reset_index(drop=True)
earthquake_links.insert(0, "지진대피소_id", range(1, len(earthquake_links) + 1))

tsunami_links = tsunami_source[["대피소명", "주소", "수용인원"]].copy()
tsunami_links["대피소_id"] = [
    shelter_id_map[(name, address)]
    for name, address in tsunami_links[["대피소명", "주소"]].itertuples(index=False, name=None)
]
tsunami_links = tsunami_links[["대피소_id", "수용인원"]].drop_duplicates().reset_index(drop=True)
tsunami_links.insert(0, "지진해일대피소_id", range(1, len(tsunami_links) + 1))


In [3]:
metadata = MetaData()

regions_table = Table(
    "regions",
    metadata,
    Column("지역_id", Integer, primary_key=True),
    Column("시도", String(20), nullable=False),
    Column("시군구", String(40)),
)

disaster_types_table = Table(
    "disaster_types",
    metadata,
    Column("재난유형_id", Integer, primary_key=True),
    Column("재난이름", String(40), nullable=False, unique=True),
)

shelters_table = Table(
    "shelters",
    metadata,
    Column("대피소_id", Integer, primary_key=True),
    Column("지역_id", Integer, ForeignKey("regions.지역_id"), nullable=False),
    Column("대피소명", String(200), nullable=False),
    Column("주소", String(255), nullable=False),
    Column("대피소유형", String(100)),
    Column("위도", Float),
    Column("경도", Float),
    Column("지역설명", String(80)),
    Column("수용인원", Integer),
    UniqueConstraint("대피소명", "주소"),
)

danger_alerts_table = Table(
    "danger_alerts",
    metadata,
    Column("위험정보_id", Integer, primary_key=True),
    Column("발표시간", DateTime),
    Column("지역_id", Integer, ForeignKey("regions.지역_id"), nullable=False),
    Column("재난유형_id", Integer, ForeignKey("disaster_types.재난유형_id"), nullable=False),
    Column("특보등급", String(20)),
    Column("해당지역", Text),
)

earthquake_shelters_table = Table(
    "earthquake_shelters",
    metadata,
    Column("지진대피소_id", Integer, primary_key=True),
    Column("대피소_id", Integer, ForeignKey("shelters.대피소_id"), nullable=False),
    Column("수용인원", Integer),
)

tsunami_shelters_table = Table(
    "tsunami_shelters",
    metadata,
    Column("지진해일대피소_id", Integer, primary_key=True),
    Column("대피소_id", Integer, ForeignKey("shelters.대피소_id"), nullable=False),
    Column("수용인원", Integer),
)

old_engine = globals().get("engine")
if old_engine is not None:
    old_engine.dispose()

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
if DB_PATH.exists():
    DB_PATH.unlink()

engine = create_engine(f"sqlite:///{DB_PATH.as_posix()}", future=True)
metadata.create_all(engine)

with engine.begin() as connection:
    connection.exec_driver_sql("PRAGMA foreign_keys = ON")
    connection.execute(insert(regions_table), to_records(regions))
    connection.execute(insert(disaster_types_table), to_records(disaster_types))
    connection.execute(insert(shelters_table), to_records(shelters))
    connection.execute(insert(danger_alerts_table), to_records(danger_alerts))
    connection.execute(insert(earthquake_shelters_table), to_records(earthquake_links))
    connection.execute(insert(tsunami_shelters_table), to_records(tsunami_links))

summary_df = pd.DataFrame(
    {
        "table_name": [
            "regions",
            "disaster_types",
            "shelters",
            "danger_alerts",
            "earthquake_shelters",
            "tsunami_shelters",
        ],
        "row_count": [
            len(regions),
            len(disaster_types),
            len(shelters),
            len(danger_alerts),
            len(earthquake_links),
            len(tsunami_links),
        ],
    }
)

print(f"DB created: {DB_PATH}")
summary_df


DB created: C:\project_dashboard\preprocessing_code\data\Emergency_shelter.db


,table_name,row_count
0,regions,73
1,disaster_types,8
2,shelters,21863
3,danger_alerts,3545
4,earthquake_shelters,2897
5,tsunami_shelters,194
